# back-fn-call-with-recipe-args — ex1: call a dispatched back_fn with (grad_out, node.array, *args, **kwargs)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `back-fn-call-with-recipe-args`. Running the final beacon cell reports progress against the `Backprop: back fn call with recipe args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: back fn call with recipe args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`back-fn-call-with-recipe-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "back-fn-call-with-recipe-args"
DD_SUBTOPIC = "Backprop: back fn call with recipe args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Back fn call with recipe args — quick refresher

Once you've dispatched the back_fn, you call it with the cached forward output, plus the original args/kwargs that the Recipe stored:

```python
grad_parent = back_fn(
    grad_out,            # dL/d(this node's output)
    node.array,          # the cached forward `out` for this node
    *node.recipe.args,   # raw positional args at call time (unboxed)
    **node.recipe.kwargs # kwargs the forward used (dim, keepdim, ...)
)
```

Three places things go wrong:
- **Forgetting `*recipe.args`** — back_fn for `multiply` needs both   inputs to compute `dL/dx = grad_out * y`. Drop them and you have   no derivative.
- **Forgetting `**recipe.kwargs`** — `sum_back` needs `dim` to   broadcast back; without it you get a shape mismatch.
- **Passing `node` instead of `node.array`** — back_fns operate on   raw torch tensors, not the MiniTensor wrapper. The whole layer   exists to keep the back_fn signature uniform across ops.

### Exercise 1 — call a dispatched back_fn with (grad_out, node.array, *args, **kwargs)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the canonical back_fn invocation pattern: call back_fn(grad_out, node.array, *recipe.args, **recipe.kwargs) so both positional and keyword args from the forward call reach the reverse pass.
> Keywords: back-fn-call, recipe-args, recipe-kwargs, splat
> ```

**KCs targeted:** `back-fn-call-with-recipe-args`, `kwargs-pass-through-recipe`

Implement `call_back_fn(back_fn, grad_out, node)` — invoke a back_fn with the canonical argument shape:

```
back_fn(
    grad_out,                     # dL/d(out)
    node.array,                   # the cached forward `out`
    *node.recipe.args,            # raw positional args (unboxed)
    **node.recipe.kwargs,         # forward kwargs (dim, keepdim, ...)
)
```

**Why every argument.**
- `grad_out`: dL/d(this node's output). The chain-rule   multiplier.
- `node.array`: the cached forward `out`. Activations like   sigmoid use it to avoid recomputation.
- `*node.recipe.args`: the original positional inputs at   call-time, unboxed. `multiply_back0` needs both `x` and `y`   to compute `dL/dx = grad_out * y`.
- `**node.recipe.kwargs`: forward keyword args. `sum_back`   needs `dim` to broadcast back; without it you get a shape   mismatch deep in the reverse pass.

**Common bugs the test catches.**
1. Passing `node` instead of `node.array` — back_fns operate on raw torch tensors, not MiniTensors.
2. Forgetting the `*` on `recipe.args` — passing the whole tuple as a single arg.
3. Forgetting the `**` on `recipe.kwargs` — `sum_back` gets called with `dim=` missing and reduces over the default axis.

Return whatever the back_fn returned (a `torch.Tensor` with the same shape as the parent at that argnum).

In [ ]:
def call_back_fn(back_fn, grad_out, node):
    # Canonical invocation — note BOTH * and ** splats.
    return back_fn(
        grad_out,
        node.array,                # raw torch.Tensor, NOT the MiniTensor
        *node.recipe.args,         # forward positional args (unboxed)
        **node.recipe.kwargs,      # forward keyword args (dim, keepdim, ...)
    )


<details><summary>Solution</summary>

```python
def call_back_fn(back_fn, grad_out, node):
    # Canonical invocation — note BOTH * and ** splats.
    return back_fn(
        grad_out,
        node.array,                # raw torch.Tensor, NOT the MiniTensor
        *node.recipe.args,         # forward positional args (unboxed)
        **node.recipe.kwargs,      # forward keyword args (dim, keepdim, ...)
    )
```

**Why the splats matter.** Without `*node.recipe.args`, you'd pass the whole tuple as one argument and `multiply_back0(grad, out, (x, y))` would crash on signature mismatch. Without `**node.recipe.kwargs`, `sum_back` would be called with no `dim=` and silently reduce over the wrong axis — the failure shows up much later as a shape mismatch.

**Why `node.array`, not `node`.** Back_fns operate on raw torch tensors so they can do tensor math directly. Passing the MiniTensor wrapper would force every back_fn body to do `out.array` first — defeating the whole point of the unboxing wrapper.

**This is the call-site half of the dispatcher.** The sibling `dispatch-back-fn-from-recipe` atom answers 'WHICH back_fn?'; this atom answers 'with WHAT args?'. The actual reverse-pass driver in `backprop` interleaves them in one tight loop, but the responsibilities are distinct.

**Why no return-type validation.** Each back_fn is responsible for returning a tensor with the correct shape (matching the parent at that argnum). The dispatcher would have to compare against each parent's shape to validate — that's pushed up to the reverse-pass loop, which already has the parent reference.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()